In [20]:
!pip install -q langchain langchain-google-genai langchain-core pydantic

In [21]:
# Cell 1: API Key
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [22]:
# Cell 2: Install LangChain packages
!pip install -q langchain langchain-google-genai

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI

def get_llm():
    return ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.7
    )

In [24]:
# ==========================================
# CELL: FAST AI VIVA GENERATOR
# ==========================================

import os
import sys
from typing import List
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ==========================================
# 1. CONFIG
# ==========================================

TOPICS = [
    "Verilog",
    "8051",
    "Digital Electronics",
    "VLSI",
    "Python",
    "Communication"
]

LEVELS = [
    "Beginner",
    "Intermediate",
    "Advanced"
]

LEVEL_TO_NUM = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

NUM_TO_LEVEL = {
    1: "Beginner",
    2: "Intermediate",
    3: "Advanced"
}

OPTION_LETTERS = ["A", "B", "C", "D"]

MODEL_NAME = "gemini-3.6-flash"


# ==========================================
# 2. LLM
# ==========================================

def get_llm():

    api_key = os.environ.get("GOOGLE_API_KEY")

    if not api_key:
        print("ERROR: GOOGLE_API_KEY not found.")
        sys.exit(1)

    return ChatGoogleGenerativeAI(
        model=MODEL_NAME
    )


# ==========================================
# 3. QUESTION SCHEMA
# ==========================================

class VivaQuestion(BaseModel):

    question: str = Field(
        description="Multiple choice viva question"
    )

    options: List[str] = Field(
        description="Exactly 4 answer options",
        min_length=4,
        max_length=4
    )

    correct_index: int = Field(
        description="Correct option index from 0 to 3",
        ge=0,
        le=3
    )

    explanation: str = Field(
        description="Short explanation of correct answer"
    )

    concept: str = Field(
        description="Short concept name"
    )

    difficulty: int = Field(
        description="1=Beginner, 2=Intermediate, 3=Advanced",
        ge=1,
        le=3
    )


class VivaQuestionSet(BaseModel):

    questions: List[VivaQuestion]


class ReportRecommendations(BaseModel):

    recommendations: List[str]


# ==========================================
# 4. PROMPT
# ==========================================

question_prompt = ChatPromptTemplate.from_template("""

You are a strict engineering and computer science
viva examiner.

Topic:
{topic}

Generate a question bank for an adaptive viva.

Generate EXACTLY:

5 Beginner questions
5 Intermediate questions
5 Advanced questions

Total = 15 questions.

Rules:

- Every question MUST be about {topic}.
- Do not include unrelated subjects.
- Each question must have exactly 4 options.
- Only ONE option must be correct.
- Give a short explanation.
- Give a short concept name.
- Mark difficulty correctly.

Beginner:
Basic definitions and fundamentals.

Intermediate:
Application, comparison and understanding.

Advanced:
Reasoning, edge cases and deeper concepts.

Do not repeat questions.

Return only the structured question set.
""")


# ==========================================
# 5. VIVA SESSION
# ==========================================

class VivaSession:

    def __init__(
        self,
        llm,
        topic,
        start_level,
        num_questions
    ):

        self.llm = llm
        self.topic = topic

        self.difficulty = LEVEL_TO_NUM[start_level]

        self.num_questions = num_questions

        self.question_chain = (
            question_prompt
            | llm.with_structured_output(VivaQuestionSet)
        )

        self.rec_chain = None

        self.questions = []

        self.asked_questions = []

        self.results = []


    # ======================================
    # GENERATE QUESTION BANK ONCE
    # ======================================

    def generate_question_bank(self):

        print("\nGenerating viva questions...")

        question_set = self.question_chain.invoke({
            "topic": self.topic
        })

        self.questions = question_set.questions

        print(
            f"✓ {len(self.questions)} questions generated."
        )


    # ======================================
    # SELECT QUESTION LOCALLY
    # ======================================

    def next_question(self):

        # Find questions matching current difficulty

        available = [
            q for q in self.questions
            if q.difficulty == self.difficulty
            and q.question not in self.asked_questions
        ]

        # If no question at current difficulty,
        # use any unused question

        if not available:

            available = [
                q for q in self.questions
                if q.question not in self.asked_questions
            ]

        # Safety check

        if not available:

            return None

        q = available[0]

        self.asked_questions.append(
            q.question
        )

        return q


    # ======================================
    # ADAPT DIFFICULTY LOCALLY
    # ======================================

    def adjust_difficulty(self, correct):

        if correct:

            self.difficulty = min(
                3,
                self.difficulty + 1
            )

        else:

            self.difficulty = max(
                1,
                self.difficulty - 1
            )


    # ======================================
    # RECORD RESULT
    # ======================================

    def record(
        self,
        question,
        concept,
        correct
    ):

        self.results.append({

            "question": question,

            "concept": concept,

            "correct": correct

        })


    # ======================================
    # GET USER ANSWER
    # ======================================

    def ask_for_choice(self):

        while True:

            raw = input(
                "Your answer (A/B/C/D): "
            ).strip().upper()

            # Accept A/B/C/D

            if raw in OPTION_LETTERS:

                return OPTION_LETTERS.index(raw)

            # Also accept 1/2/3/4

            if raw in ["1", "2", "3", "4"]:

                return int(raw) - 1

            print(
                "Please enter A, B, C, D "
                "or 1, 2, 3, 4."
            )


    # ======================================
    # RUN VIVA
    # ======================================

    def run(self):

        print(
            f"\nStarting viva on '{self.topic}' "
            f"(starting level: "
            f"{NUM_TO_LEVEL[self.difficulty]})\n"
        )

        # ONE GEMINI CALL

        self.generate_question_bank()

        # Ask questions locally

        for i in range(
            1,
            self.num_questions + 1
        ):

            q = self.next_question()

            if q is None:

                print(
                    "\nNo more questions available."
                )

                break

            print(
                f"\nQ{i}. "
                f"[{NUM_TO_LEVEL[q.difficulty]}] "
                f"{q.question}"
            )

            for letter, option in zip(
                OPTION_LETTERS,
                q.options
            ):

                print(
                    f"   {letter}. {option}"
                )

            chosen_index = self.ask_for_choice()

            correct = (
                chosen_index ==
                q.correct_index
            )

            if correct:

                print("\n✅ Correct!")

            else:

                correct_letter = (
                    OPTION_LETTERS[
                        q.correct_index
                    ]
                )

                print(
                    f"\n❌ Incorrect."
                )

                print(
                    f"Correct answer: "
                    f"{correct_letter}. "
                    f"{q.options[q.correct_index]}"
                )

            print(
                f"Explanation: "
                f"{q.explanation}"
            )

            self.record(
                q.question,
                q.concept,
                correct
            )

            self.adjust_difficulty(
                correct
            )

            if correct:

                print(
                    "Next question will be harder."
                )

            else:

                print(
                    "Next question will be easier."
                )

        self.print_report()


    # ======================================
    # FINAL REPORT
    # ======================================

    def print_report(self):

        total = len(self.results)

        correct_count = sum(
            1
            for r in self.results
            if r["correct"]
        )

        pct = (
            round(
                100 *
                correct_count /
                total
            )
            if total
            else 0
        )

        strong_concepts = sorted({
            r["concept"]
            for r in self.results
            if r["correct"]
        })

        weak_concepts = sorted({
            r["concept"]
            for r in self.results
            if not r["correct"]
        })


        print("\n")
        print("━" * 30)

        print(
            "       VIVA REPORT"
        )

        print("━" * 30)

        print(
            f"Topic: {self.topic}\n"
        )

        print(
            f"Questions:  {total}"
        )

        print(
            f"Correct:    {correct_count}"
        )

        print(
            f"Score:      {pct}%\n"
        )


        print("Strong Areas:")

        if strong_concepts:

            for concept in strong_concepts:

                print(
                    f"✓ {concept}"
                )

        else:

            print(
                "(none yet)"
            )


        print("\nWeak Areas:")

        if weak_concepts:

            for concept in weak_concepts:

                print(
                    f"⚠ {concept}"
                )

        else:

            print(
                "(none — great work!)"
            )


        print("\nRecommended:")

        if weak_concepts:

            for concept in weak_concepts:

                print(
                    f"→ Revise {concept}"
                )

        else:

            print(
                "→ Try the next difficulty level"
            )

        print("━" * 30)


# ==========================================
# 6. MENU
# ==========================================

def choose_from_list(
    prompt,
    options
):

    print(prompt)

    for i, option in enumerate(
        options,
        1
    ):

        print(
            f"  {i}. {option}"
        )

    while True:

        choice = input(
            f"Choose 1-{len(options)}: "
        ).strip()

        if (
            choice.isdigit()
            and
            1 <= int(choice) <= len(options)
        ):

            return options[
                int(choice) - 1
            ]

        print(
            "Invalid choice, try again."
        )


# ==========================================
# 7. MAIN
# ==========================================

def main():

    print("=" * 30)

    print(
        "   AI VIVA GENERATOR (MCQ)"
    )

    print("=" * 30)


    topic = choose_from_list(
        "\nSelect a topic:",
        TOPICS
    )


    level = choose_from_list(
        "\nSelect starting difficulty:",
        LEVELS
    )


    while True:

        n = input(
            "\nHow many questions? "
        ).strip()

        if (
            n.isdigit()
            and int(n) > 0
        ):

            num_questions = int(n)

            break

        print(
            "Please enter a positive number."
        )


    llm = get_llm()


    session = VivaSession(
        llm,
        topic,
        level,
        num_questions
    )


    session.run()

In [25]:
main()

   AI VIVA GENERATOR (MCQ)

Select a topic:
  1. Verilog
  2. 8051
  3. Digital Electronics
  4. VLSI
  5. Python
  6. Communication
Choose 1-6: GG
Invalid choice, try again.
Choose 1-6: 5

Select starting difficulty:
  1. Beginner
  2. Intermediate
  3. Advanced
Choose 1-3: 1

How many questions? 3

Starting viva on 'Python' (starting level: Beginner)


Generating viva questions...
✓ 15 questions generated.

Q1. [Beginner] What is the output of type([]) in Python?
   A. <class 'tuple'>
   B. <class 'list'>
   C. <class 'dict'>
   D. <class 'set'>
Your answer (A/B/C/D): A

❌ Incorrect.
Correct answer: B. <class 'list'>
Explanation: The syntax [] denotes a list literal in Python, so type([]) returns <class 'list'>.
Next question will be easier.

Q2. [Beginner] Which keyword is used to define a function in Python?
   A. function
   B. def
   C. func
   D. define
Your answer (A/B/C/D): A

❌ Incorrect.
Correct answer: B. def
Explanation: The 'def' keyword is used to define functions in Pyt